In [1]:
# Import necessary packages
import pandas as pd
import glob
import os
import numpy as np
from scipy.stats import fisher_exact

In [2]:
# Read data, here the ClinVar mut mapped to GI* scores, and the top 10% of protein-TRN
df = pd.read_csv("clinvar_mapped.csv")
top = pd.read_csv('../top_10percent_proteins_withtoptrns_scores.csv')

In [3]:
# Set cutoffs for freq and rare, apply to df
top_cutoff = df['count'].mean() + df['count'].std()
bottom_cutoff = df['count'].mean() - df['count'].std()
conditions = [
    df['count'] > top_cutoff,
    df['count'] < bottom_cutoff
]

choices = ['freq', 'rare']

df['freq'] = np.select(conditions, choices, default='intermediate')

# Print counts
df['freq'].value_counts()

freq
intermediate    1679050
freq             153514
rare               2292
Name: count, dtype: int64

In [4]:
df.to_csv('clinvar_frequency_df.csv',index=False)

In [5]:
# Look at top 10% and print counts
sub = df[df['protein'].isin(top['protein'].tolist())]
sub['freq'].value_counts()

freq
intermediate    83183
freq             9352
rare              111
Name: count, dtype: int64

In [6]:
# Proportion
print("percentage of data: ", 9352/83183 * 100)

percentage of data:  11.242681797963526


In [7]:
# Enrichment and p-val
top_yes = 9352
top_no = 83183 + 111

non_top_yes = 153514 - top_yes
non_top_no = 1679050 + 2292 - top_no

table = [
    [top_yes, top_no],
    [non_top_yes, non_top_no]
]

odds_ratio, pval = fisher_exact(table, alternative='greater')

print("Odds ratio:", odds_ratio)
print("P-value:", pval)

Odds ratio: 1.2446000117416056
P-value: 1.932846028756352e-80
